# E2 – Forward Pass from Scratch
**Introduction to Deep Learning, THWS**

In this session you will build a neural network forward pass from the ground up — starting from a single scalar function and working your way up to a batched, multi-layer network.

**How to work:** All your implementations go in `linear.py`. Edit that file in your editor, then run the cells here to test and visualise your work. You do not need to modify this notebook.

**Structure:**
1. Scalar linear function and scalar ReLU
2. Composition and stacking → piecewise linear functions
3. Vector input, scalar output
4. Batching: loop vs. vectorised
5. Stateful objects: `LinearLayer` and `MLP`

In [ ]:
import time
import torch
%matplotlib inline

import linear
import plotting

---
## Block 1 – Scalar linear function and scalar ReLU

The simplest possible setting: one number in, one number out.

**Task 1.1** Implement `linear_scalar(x, theta_1, theta_0)` in `linear.py`.
The function should return $\theta_1 \cdot x + \theta_0$.

**Task 1.2** Implement `relu_scalar(x)` in `linear.py`.
The function should return $\max(0, x)$.

In [ ]:
# Reload your module after each edit
import importlib; importlib.reload(linear)
from linear import linear_scalar, relu_scalar

# Sanity checks
assert linear_scalar(2.0, 3.0, -1.0) == 5.0,  "linear_scalar(2, 3, -1) should be 5"
assert linear_scalar(0.0, 3.0, -1.0) == -1.0, "linear_scalar(0, 3, -1) should be -1"
assert relu_scalar(2.5)  == 2.5, "relu_scalar(2.5) should be 2.5"
assert relu_scalar(-1.0) == 0.0, "relu_scalar(-1) should be 0"
print("All checks passed.")

In [ ]:
importlib.reload(plotting)
fig = plotting.plot_linear_scalar()

# Question: what can a scalar linear function NOT represent?

In [ ]:
fig = plotting.plot_relu_scalar()

# Question: where does the output change behaviour, and why?

---
## Block 2 – Composition and stacking → piecewise linear functions

A single linear function can only represent a straight line. What happens when we compose and stack them?

**Task 2.1** Implement `relu_unit(x, theta_1, theta_0)` in `linear.py` using your functions from Block 1.

**Task 2.2** Implement `stack_layers(x, params)` in `linear.py`.
`params` is a list of `(theta_1, theta_0)` tuples. Apply a linear transformation at each layer and a ReLU after every layer except the last.

In [ ]:
importlib.reload(linear)
from linear import relu_unit, stack_layers

# relu_unit: linear then ReLU
assert relu_unit(1.0, 2.0, -0.5) == 1.5,  "relu_unit(1, 2, -0.5) = relu(1.5) = 1.5"
assert relu_unit(-1.0, 2.0, -0.5) == 0.0, "relu_unit(-1, 2, -0.5) = relu(-2.5) = 0"

# stack_layers: two-layer example, manual check
# Layer 0: relu(2.0 * 1.0 - 0.5) = relu(1.5) = 1.5
# Layer 1: 1.0 * 1.5 + 0.0 = 1.5  (no ReLU on last layer)
assert stack_layers(1.0, [(2.0, -0.5), (1.0, 0.0)]) == 1.5
print("All checks passed.")

In [ ]:
importlib.reload(plotting)
fig = plotting.plot_stacking()

# Question: is the output always piecewise linear, regardless of depth?
# Question: can you find a set of parameters that produces a constant output?

---
## Block 3 – Vector input, scalar output

Real data is rarely one-dimensional. The natural generalisation is to replace the scalar weight $\theta_1$ with a weight *vector* and compute a dot product.

**Task 3.1** Implement `linear_vector(x, theta_1, theta_0)` in `linear.py` using `torch.dot`.

**Task 3.2** Implement `relu_tensor(x)` in `linear.py` using `torch.clamp`.

In [ ]:
importlib.reload(linear)
from linear import linear_vector, relu_tensor

x_vec     = torch.tensor([1.0, 2.0])
theta_1   = torch.tensor([0.5, -1.0])
theta_0   = 0.3
# 0.5*1 + (-1.0)*2 + 0.3 = 0.5 - 2.0 + 0.3 = -1.2
assert abs(linear_vector(x_vec, theta_1, theta_0).item() - (-1.2)) < 1e-5

t = torch.tensor([-1.0, 0.0, 2.0])
expected = torch.tensor([0.0, 0.0, 2.0])
assert torch.allclose(relu_tensor(t), expected)
print("All checks passed.")

In [ ]:
importlib.reload(plotting)
fig = plotting.plot_surface()

# Question: what does the ReLU surface look like geometrically?
# Question: can this single unit fit an arbitrary function over 2D inputs?

---
## Block 4 – Batching: loop vs. vectorised

We never have just one observation — we have $N$ of them. Computing the forward pass one sample at a time with a Python loop is correct but slow. Matrix multiplication lets us process the entire batch at once.

**Task 4.1** Implement `forward_loop(X, theta_1, theta_0)` in `linear.py`:  
loop over rows of `X`, compute a dot product for each one.

**Task 4.2** Implement `forward_vectorised(X, theta_1, theta_0)` in `linear.py`:  
compute the same result in a single line using the `@` operator.

In [ ]:
importlib.reload(linear)
from linear import forward_loop, forward_vectorised

theta_1_b = torch.tensor([1.0, 0.5])
theta_0_b = torch.tensor(0.3)
N = 100_000
X = torch.randn(N, 2)

t0 = time.time(); out_loop = forward_loop(X, theta_1_b, theta_0_b); t_loop = time.time() - t0
t0 = time.time(); out_vec  = forward_vectorised(X, theta_1_b, theta_0_b); t_vec  = time.time() - t0

assert torch.allclose(out_loop, out_vec, atol=1e-5), "Outputs do not match!"
print(f"Loop:        {t_loop:.4f} s")
print(f"Vectorised:  {t_vec:.4f} s")
print(f"Speed-up:    {t_loop / t_vec:.1f}x")

In [ ]:
importlib.reload(plotting)
fig = plotting.plot_timing(t_loop, t_vec, N)

---
## Block 5 – Stateful objects: `LinearLayer` and `MLP`

So far, parameters `theta_1` and `theta_0` are loose variables that must be passed into every function call. As networks grow deeper this becomes unwieldy — imagine threading six pairs of parameters through a chain of function calls.

A better approach: encapsulate parameters inside an object that carries them around.

**Task 5.1** Implement the `LinearLayer` class in `linear.py`.  
The constructor stores `theta_1` and `theta_0`; `forward` computes the linear transformation.

**Fast finisher — Task 5.2** Implement the `MLP` class in `linear.py`.  
It holds a list of `LinearLayer` objects and applies ReLU between layers.

In [ ]:
importlib.reload(linear)
from linear import LinearLayer

theta_1_l = torch.randn(3, 2)
theta_0_l = torch.randn(3)
X_test    = torch.randn(5, 2)

layer      = LinearLayer(theta_1_l, theta_0_l)
out_layer  = layer.forward(X_test)
out_manual = X_test @ theta_1_l.T + theta_0_l

assert out_layer.shape == torch.Size([5, 3]),              "Wrong output shape"
assert torch.allclose(out_layer, out_manual, atol=1e-5),   "Output does not match manual computation"
print(f"Output shape: {out_layer.shape}")
print("All checks passed.")

In [ ]:
# Fast finisher: MLP
from linear import MLP

layer1 = LinearLayer(torch.randn(4, 2), torch.randn(4))
layer2 = LinearLayer(torch.randn(1, 4), torch.randn(1))
mlp    = MLP([layer1, layer2])

X_mlp    = torch.randn(5, 2)
out_mlp  = mlp.forward(X_mlp)

assert out_mlp.shape == torch.Size([5, 1]), "Wrong output shape"
print(f"Input shape:  {X_mlp.shape}")
print(f"Output shape: {out_mlp.shape}")
print("All checks passed.")

# Question: in L5 we will replace LinearLayer with nn.Module and MLP with nn.Sequential.
# What do you expect will be different? What will stay the same?